In [1]:
# ================================================================
# 📦 Maharashtra Water Footprint Synthetic Dataset (AI-Powered)
# ================================================================

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import random
import pandas as pd
import numpy as np
from openai import OpenAI

# --- Step 1: Setup output path ---
save_path = "/content/drive/MyDrive/WaterFootprintData/"
os.makedirs(save_path, exist_ok=True)

# --- Step 2: Initialize OpenRouter client ---
OPENROUTER_API_KEY = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# --- Step 3: Define regions and crops ---
districts = [
    "Pune", "Nagpur", "Nashik", "Aurangabad", "Solapur", "Ratnagiri",
    "Raigad", "Thane", "Satara", "Sangli", "Amravati", "Kolhapur",
    "Ahmednagar", "Chandrapur"
]

crops = [
    "Sugarcane", "Soybean", "Cotton", "Rice", "Jowar",
    "Wheat", "Tur", "Maize", "Groundnut", "Banana", "Grapes", "Onions"
]

# --- Step 4: Fetch AI water footprint per crop ---
def get_water_footprint_value(crop_name):
    """Fetch average water footprint (L/kg) for the given crop in Maharashtra."""
    try:
        prompt = f"""
        You are an agricultural water usage expert in India.
        Provide the *average water footprint in liters per kilogram (L/kg)*
        for the crop '{crop_name}' grown in Maharashtra, India.
        Return only a numeric value (no units or explanation).
        """

        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "You provide precise factual data only."},
                {"role": "user", "content": prompt},
            ],
        )

        value_text = response.choices[0].message.content.strip()
        value = ''.join(ch for ch in value_text if ch.isdigit() or ch == '.')
        return float(value) if value else None

    except Exception as e:
        print(f"⚠ API error for {crop_name}: {e}")
        return None


# --- Step 5: Fetch season per crop ---
def get_crop_season(crop_name):
    """Fetch the main growing season (Rabi, Kharif, or Zaid) for a given crop."""
    try:
        prompt = f"""
        In Maharashtra, India, what is the main growing season for the crop '{crop_name}'?
        Choose only one from: Rabi, Kharif, or Zaid.
        Give only the name of the season.
        """

        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "Answer with only one season name."},
                {"role": "user", "content": prompt},
            ],
        )

        season_text = response.choices[0].message.content.strip().capitalize()
        if season_text in ["Rabi", "Kharif", "Zaid"]:
            return season_text
        return random.choice(["Rabi", "Kharif", "Zaid"])  # fallback

    except Exception as e:
        print(f"⚠ API error fetching season for {crop_name}: {e}")
        return random.choice(["Rabi", "Kharif", "Zaid"])


# --- Step 6: Check if crop is grown in district ---
def is_crop_grown_in_district(crop_name, district_name):
    """Uses AI to verify if a crop is typically cultivated in a district."""
    try:
        prompt = f"""
        In Maharashtra, is the crop '{crop_name}' commonly grown in the district '{district_name}'?
        Answer strictly with 'Yes' or 'No'.
        """

        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "Answer only Yes or No."},
                {"role": "user", "content": prompt},
            ],
        )

        answer = response.choices[0].message.content.strip().lower()
        return "yes" in answer

    except Exception as e:
        print(f"⚠ API error crop-region check ({crop_name}, {district_name}): {e}")
        return True  # assume yes to avoid data gaps


# --- Step 7: Build water + season mapping ---
print("🔹 Fetching AI-based crop parameters...\n")
crop_water_map = {}
crop_season_map = {}

for crop in crops:
    val = get_water_footprint_value(crop)
    if val is None:
        val = np.random.uniform(1200, 4000)
    crop_water_map[crop] = round(val, 2)

    season = get_crop_season(crop)
    crop_season_map[crop] = season

    print(f"✅ {crop}: {val:.2f} L/kg, Season: {season}")
    time.sleep(1.2)

print("\n✅ All crop base values and seasons fetched.\n")


# --- Step 8: Generate region-aware dataset (Permutation-based) ---
records = []
print("🔹 Generating dataset from all valid Crop × District × Irrigation_Type combinations...\n")

irrigation_types = ["Canal", "Drip", "Rainfed", "Well"]

for district in districts:
    for crop in crops:
        # Skip if not typically grown in this district
        if not is_crop_grown_in_district(crop, district):
            continue

        for irrigation_type in irrigation_types:
            base_value = crop_water_map[crop]
            avg_value = base_value * np.random.uniform(0.9, 1.1)

            if avg_value > 3000:
                category = "High"
            elif avg_value > 1500:
                category = "Medium"
            else:
                category = "Low"

            season = crop_season_map[crop]

            records.append({
                "State": "Maharashtra",
                "District": district,
                "Crop": crop,
                "Season": season,
                "Irrigation_Type": irrigation_type,
                "Avg_Water_Requirement(L_per_kg)": round(avg_value, 2),
                "Category": category
            })

    print(f"✅ Completed combinations for {district}")

# --- Step 9: Save the dataset ---
df = pd.DataFrame(records)
print(f"\n✅ Total valid records generated: {len(df)}")

# Save locally first
local_path = "/content/maharashtra_water_footprint_ai_region_aware.csv"
df.to_csv(local_path, index=False)
print(f"📁 Saved locally at: {local_path}")

# Copy to Drive
final_path = os.path.join(save_path, "maharashtra_water_footprint_ai_region_aware.csv")
os.system(f"cp '{local_path}' '{final_path}'")
print(f"✅ Successfully copied to Drive: {final_path}")

print("\nSample Preview:")
print(df.head(10))


Mounted at /content/drive
🔹 Fetching AI-based crop parameters...

✅ Sugarcane: 2500.00 L/kg, Season: Rabi
✅ Soybean: 1300.00 L/kg, Season: Kharif
✅ Cotton: 7000.00 L/kg, Season: Zaid
✅ Rice: 1500.00 L/kg, Season: Kharif
✅ Jowar: 1200.00 L/kg, Season: Kharif
✅ Wheat: 1300.00 L/kg, Season: Rabi
✅ Tur: 6001200.00 L/kg, Season: Kharif
✅ Maize: 520.00 L/kg, Season: Zaid
✅ Groundnut: 1200.00 L/kg, Season: Rabi
✅ Banana: 10001200.00 L/kg, Season: Kharif
✅ Grapes: 1100.00 L/kg, Season: Kharif
✅ Onions: 240.00 L/kg, Season: Zaid

✅ All crop base values and seasons fetched.

🔹 Generating dataset from all valid Crop × District × Irrigation_Type combinations...

✅ Completed combinations for Pune
✅ Completed combinations for Nagpur
✅ Completed combinations for Nashik
✅ Completed combinations for Aurangabad
✅ Completed combinations for Solapur
✅ Completed combinations for Ratnagiri
✅ Completed combinations for Raigad
✅ Completed combinations for Thane
✅ Completed combinations for Satara
✅ Completed 

In [2]:
# ================================================================
# 📦 Maharashtra Water Footprint Synthetic Dataset (API-driven, validated)
# ================================================================

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import random
import json
import re
import pandas as pd
import numpy as np
from openai import OpenAI

# --- Step 1: Setup output path ---
save_path = "/content/drive/MyDrive/WaterFootprintData/"
os.makedirs(save_path, exist_ok=True)

# --- Step 2: Initialize OpenRouter client ---
OPENROUTER_API_KEY = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# --- Step 3: Define regions and crops ---
districts = [
    "Pune", "Nagpur", "Nashik", "Aurangabad", "Solapur", "Ratnagiri",
    "Raigad", "Thane", "Satara", "Sangli", "Amravati", "Kolhapur",
    "Ahmednagar", "Chandrapur"
]

crops = [
    "Sugarcane", "Soybean", "Cotton", "Rice", "Jowar",
    "Wheat", "Tur", "Maize", "Groundnut", "Banana", "Grapes", "Onions"
]

# --- Helper: extract JSON blob from model text ---
def extract_json_from_text(text):
    """
    Try to find a JSON object in text and load it.
    Returns a dict or None.
    """
    # first try to find {...} substring
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        blob = match.group(0)
        try:
            return json.loads(blob)
        except Exception:
            # try to "fix" common issues: replace single quotes, trailing commas
            cleaned = blob.replace("'", "\"")
            cleaned = re.sub(r",\s*}", "}", cleaned)
            cleaned = re.sub(r",\s*]", "]", cleaned)
            try:
                return json.loads(cleaned)
            except Exception:
                return None
    # if no JSON, try to find a single number in the text
    num_match = re.search(r"[-+]?\d{1,3}(?:[,\d]{0,})?(?:\.\d+)?", text)
    if num_match:
        num_str = num_match.group(0).replace(",", "")
        try:
            return {"value": float(num_str)}
        except Exception:
            return None
    return None

# --- Step 4: Fetch AI water footprint per crop (robust, validation + retries) ---
def get_water_footprint_value(crop_name, max_retries=4, min_val=5.0, max_val=10000.0, sleep_between=1.0):
    """
    API-only retrieval of average water footprint (L/kg) for crop_name in Maharashtra.
    Repeats up to `max_retries` times if the model returns invalid output.
    Returns a float value if valid, otherwise returns None.
    """
    system_msg = (
        "You are an expert in agricultural water-use. Answer precisely and ONLY in JSON. "
        "Do NOT include extra explanation. "
        "The JSON MUST be exactly: {\"value\": <number>} where <number> is the average water footprint in liters per kilogram (L/kg). "
        "If you cannot provide a numeric estimate, respond with {\"value\": null}."
    )

    user_msg = (
        f"In Maharashtra, India, what is the average water footprint in liters per kilogram (L/kg) "
        f"for the crop '{crop_name}'? Return ONLY the JSON exactly as described."
    )

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model="mistralai/mistral-7b-instruct",
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=120,
            )

            raw = response.choices[0].message.content.strip()
            parsed = extract_json_from_text(raw)

            if parsed is None:
                # nothing parseable — retry
                print(f"⚠ parse fail (attempt {attempt}) for {crop_name} — raw: {raw!r}")
                time.sleep(sleep_between)
                continue

            # parsed could be {'value': number} or have other keys
            if "value" not in parsed:
                # if JSON present but no 'value' key, try to find numeric entries
                # attempt to locate any numeric field
                numbers = [v for v in parsed.values() if isinstance(v, (int, float))]
                if numbers:
                    value = float(numbers[0])
                else:
                    print(f"⚠ no 'value' in JSON (attempt {attempt}) for {crop_name} — json: {parsed}")
                    time.sleep(sleep_between)
                    continue
            else:
                value = parsed["value"]

            if value is None:
                # model explicitly said it cannot provide a number
                print(f"⚠ model returned null value for {crop_name} (attempt {attempt})")
                time.sleep(sleep_between)
                continue

            # validate type and numeric range
            try:
                val = float(value)
            except Exception:
                print(f"⚠ non-numeric 'value' returned for {crop_name} (attempt {attempt}): {value!r}")
                time.sleep(sleep_between)
                continue

            # sensible bounds check
            if not (min_val <= val <= max_val):
                print(
                    f"⚠ value out of bounds for {crop_name} (attempt {attempt}): {val} L/kg. "
                    f"Expected between {min_val} and {max_val}."
                )
                time.sleep(sleep_between)
                continue

            # Good value
            return round(val, 2)

        except Exception as e:
            print(f"⚠ API error on attempt {attempt} for {crop_name}: {e}")
            time.sleep(sleep_between)

    # If we reach here, all retries failed
    print(f"✖ Failed to obtain a valid water-footprint for {crop_name} after {max_retries} attempts. Setting as NaN.")
    return None


# --- Step 5: Fetch season per crop ---
def get_crop_season(crop_name):
    """Fetch the main growing season (Rabi, Kharif, or Zaid) for a given crop."""
    try:
        prompt = f"""
        In Maharashtra, India, what is the main growing season for the crop '{crop_name}'?
        Choose only one from: Rabi, Kharif, or Zaid.
        Give only the name of the season in JSON: {{"season": "Rabi"}}
        """
        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "Answer with JSON containing only the season name."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=40,
        )

        raw = response.choices[0].message.content.strip()
        parsed = extract_json_from_text(raw)
        if parsed and "season" in parsed:
            season_text = str(parsed["season"]).strip().capitalize()
            if season_text in ["Rabi", "Kharif", "Zaid"]:
                return season_text

        # fallback to a single-word parse if JSON not provided
        season_text = re.search(r"\b(Rabi|Kharif|Zaid)\b", raw, flags=re.IGNORECASE)
        if season_text:
            return season_text.group(0).capitalize()

        return random.choice(["Rabi", "Kharif", "Zaid"])
    except Exception as e:
        print(f"⚠ API error fetching season for {crop_name}: {e}")
        return random.choice(["Rabi", "Kharif", "Zaid"])


# --- Step 6: Check if crop is grown in district ---
def is_crop_grown_in_district(crop_name, district_name):
    """Uses AI to verify if a crop is typically cultivated in a district."""
    try:
        prompt = f"""
        In Maharashtra, is the crop '{crop_name}' commonly grown in the district '{district_name}'?
        Answer strictly in JSON: {{"grown": "Yes"}} or {{"grown": "No"}} (Only Yes or No).
        """
        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "Answer only with exactly the JSON object {\"grown\":\"Yes\"} or {\"grown\":\"No\"}."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=30,
        )

        raw = response.choices[0].message.content.strip()
        parsed = extract_json_from_text(raw)
        if parsed and "grown" in parsed:
            return str(parsed["grown"]).strip().lower().startswith("y")
        # fallback parse
        return "yes" in raw.lower()
    except Exception as e:
        print(f"⚠ API error crop-region check ({crop_name}, {district_name}): {e}")
        return True  # assume yes to avoid data gaps


# --- Step 7: Build water + season mapping (API-driven, no hardcoded fallback) ---
print("Fetching AI-based crop parameters...\n")
crop_water_map = {}
crop_season_map = {}

for crop in crops:
    val = get_water_footprint_value(crop)
    if val is None:
        # keep NaN so user can detect/API issues later
        crop_water_map[crop] = np.nan
    else:
        crop_water_map[crop] = val

    season = get_crop_season(crop)
    crop_season_map[crop] = season

    if val is None:
        print(f"⚠ {crop}: value unavailable (NaN), Season: {season}")
    else:
        print(f"{crop}: {val:.2f} L/kg, Season: {season}")
    time.sleep(1.2)

print("\nAll crop base values and seasons fetched.\n")


# --- Step 8: Generate region-aware dataset (Permutation-based) ---
records = []
print("Generating dataset from all valid Crop × District × Irrigation_Type combinations...\n")

irrigation_types = ["Canal", "Drip", "Rainfed", "Well"]

for district in districts:
    for crop in crops:
        # Skip if not typically grown in this district
        if not is_crop_grown_in_district(crop, district):
            continue

        for irrigation_type in irrigation_types:
            base_value = crop_water_map.get(crop, np.nan)

            # if base_value is NaN (API failed), then Avg will remain NaN
            if pd.isna(base_value):
                avg_value = np.nan
            else:
                avg_value = base_value * np.random.uniform(0.9, 1.1)

            if pd.isna(avg_value):
                category = "Unknown"
            else:
                if avg_value > 3000:
                    category = "High"
                elif avg_value > 1500:
                    category = "Medium"
                else:
                    category = "Low"

            season = crop_season_map.get(crop, "")

            records.append({
                "State": "Maharashtra",
                "District": district,
                "Crop": crop,
                "Season": season,
                "Irrigation_Type": irrigation_type,
                "Avg_Water_Requirement(L_per_kg)": round(avg_value, 2) if not pd.isna(avg_value) else np.nan,
                "Category": category
            })

    print(f"Completed combinations for {district}")

# --- Step 9: Save the dataset ---
df = pd.DataFrame(records)
print(f"\nTotal valid records generated: {len(df)}")

# Save locally first
local_path = "/content/maharashtra_water_footprint_ai_region_aware.csv"
df.to_csv(local_path, index=False)
print(f"Saved locally at: {local_path}")

# Copy to Drive
final_path = os.path.join(save_path, "maharashtra_water_footprint_ai_region_aware.csv")
os.system(f"cp '{local_path}' '{final_path}'")
print(f"Successfully copied to Drive: {final_path}")

print("\nSample Preview:")
print(df.head(10))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fetching AI-based crop parameters...

Sugarcane: 1500.00 L/kg, Season: Zaid
Soybean: 1500.00 L/kg, Season: Zaid
Cotton: 7900.00 L/kg, Season: Zaid
Rice: 2500.00 L/kg, Season: Rabi
Jowar: 1200.00 L/kg, Season: Kharif
Wheat: 1300.00 L/kg, Season: Kharif
Tur: 1200.00 L/kg, Season: Kharif
Maize: 1200.00 L/kg, Season: Rabi
Groundnut: 1500.00 L/kg, Season: Rabi
Banana: 790.00 L/kg, Season: Rabi
Grapes: 1000.00 L/kg, Season: Rabi
Onions: 245.00 L/kg, Season: Kharif

All crop base values and seasons fetched.

Generating dataset from all valid Crop × District × Irrigation_Type combinations...

Completed combinations for Pune
Completed combinations for Nagpur
Completed combinations for Nashik
Completed combinations for Aurangabad
Completed combinations for Solapur
Completed combinations for Ratnagiri
Completed combinations for Raigad
Completed combinations for Thane
Co

In [4]:
# ================================================================
# 📦 Maharashtra Water Footprint Synthetic Dataset (API-driven, ALL FACTORS FETCHED VIA AI)
# ================================================================

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import random
import json
import re
import pandas as pd
import numpy as np
from openai import OpenAI
from collections import defaultdict

# --- Step 1: Setup output path ---
save_path = "/content/drive/MyDrive/WaterFootprintData/"
os.makedirs(save_path, exist_ok=True)

# --- Step 2: Initialize OpenRouter client ---
# NOTE: Using the API key provided in the original code.
OPENROUTER_API_KEY = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# --- Step 3: Define regions and crops ---
districts = [
    "Pune", "Nagpur", "Nashik", "Aurangabad", "Solapur", "Ratnagiri",
    "Raigad", "Thane", "Satara", "Sangli", "Amravati", "Kolhapur",
    "Ahmednagar", "Chandrapur"
]

crops = [
    "Sugarcane", "Soybean", "Cotton", "Rice", "Jowar",
    "Wheat", "Tur", "Maize", "Groundnut", "Banana", "Grapes", "Onions"
]

irrigation_types = ["Canal", "Drip", "Rainfed", "Well"]

# --- Helper: extract JSON blob from model text (Improved for robustness) ---
def extract_json_from_text(text):
    """Try to find a JSON object in text and load it. Returns a dict or None."""
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        blob = match.group(0)
        try:
            return json.loads(blob)
        except Exception:
            # Attempt to fix common issues
            cleaned = blob.replace("'", "\"")
            cleaned = re.sub(r",\s*}", "}", cleaned)
            cleaned = re.sub(r",\s*]", "]", cleaned)
            try:
                return json.loads(cleaned)
            except Exception:
                return None
    return None

# --- NEW Helper: Robust API call for single/multiple numerical values ---
def get_numerical_data(user_msg, system_msg, expected_keys, max_retries=4, sleep_between=1.5):
    """
    Generalized function for API retrieval of numerical data based on a strict JSON schema.
    Returns a dictionary of results or None.
    """
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model="mistralai/mistral-7b-instruct",
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=200,
            )

            raw = response.choices[0].message.content.strip()
            parsed = extract_json_from_text(raw)

            if parsed is None:
                print(f"⚠ Parse fail (attempt {attempt}) for prompt: {user_msg[:30]}...")
                time.sleep(sleep_between)
                continue

            # Validate all expected keys are present and numeric
            result = {}
            is_valid = True
            for key in expected_keys:
                if key not in parsed:
                    print(f"⚠ Missing key '{key}' (attempt {attempt}) in JSON: {parsed}")
                    is_valid = False
                    break

                value = parsed[key]
                try:
                    # Convert to float and round
                    result[key] = round(float(value), 2)
                except (ValueError, TypeError):
                    print(f"⚠ Non-numeric value for '{key}' (attempt {attempt}): {value!r}")
                    is_valid = False
                    break

            if is_valid:
                return result

        except Exception as e:
            print(f"⚠ API error on attempt {attempt}: {e}")
            time.sleep(sleep_between)

    print(f"✖ Failed to obtain valid numerical data after {max_retries} attempts.")
    return None

# --- Step 4A: Fetch AI base water footprint (L/kg) ---
def get_base_water_footprint_value(crop_name):
    """Retrieves the base L/kg for a crop using AI."""
    system_msg = (
        "You are an expert in agricultural water-use. Answer precisely and ONLY in JSON. "
        "The JSON MUST be exactly: {\"base_value\": <number>}. "
        "Provide the average water footprint in liters per kilogram (L/kg) for this crop (e.g., Rice is ~2500 L/kg)."
    )
    user_msg = f"Provide the base water footprint (L/kg) for the crop '{crop_name}'."

    data = get_numerical_data(user_msg, system_msg, ["base_value"])
    return data["base_value"] if data else np.nan

# --- Step 4B: Fetch AI irrigation factor multipliers ---
def get_irrigation_multipliers(irrigation_type):
    """Retrieves the min/max efficiency factors for an irrigation type using AI."""
    system_msg = (
        "You are an expert in irrigation efficiency. Answer precisely and ONLY in JSON. "
        "The JSON MUST be exactly: {\"min_factor\": <number>, \"max_factor\": <number>}. "
        "Provide a min and max multiplier (e.g., 0.75 to 1.35) that represents the efficiency of this irrigation type. "
        "Efficiency > 1.0 means higher L/kg (less efficient or higher reliance on blue water)."
    )
    user_msg = f"Provide the min and max efficiency factor (multiplier) for '{irrigation_type}' irrigation."

    data = get_numerical_data(user_msg, system_msg, ["min_factor", "max_factor"])
    # Default to a neutral range if API fails to ensure script runs, though this is soft-hardcoding a fallback
    return (data["min_factor"], data["max_factor"]) if data else (0.95, 1.05)

# --- Step 4C: Fetch AI district stress index ---
def get_district_stress_index(district_name):
    """Retrieves the regional stress index for a district using AI."""
    system_msg = (
        "You are an expert in regional water stress and yield variability in Maharashtra, India. Answer precisely and ONLY in JSON. "
        "The JSON MUST be exactly: {\"factor\": <number>}. "
        "Provide a single multiplier (factor) between 0.85 and 1.20 for the district's average agricultural water stress. "
        "Higher factor (e.g., 1.15) means drier region, lower yield per drop, or higher blue water need."
    )
    user_msg = f"Provide the regional water stress factor for the district '{district_name}' in Maharashtra."

    data = get_numerical_data(user_msg, system_msg, ["factor"])
    return data["factor"] if data else 1.0

# --- Step 5: Fetch AI crop season (used for context) (UNCHANGED) ---
def get_crop_season(crop_name):
    """Fetch the main growing season (Rabi, Kharif, or Zaid) for a given crop using AI."""
    try:
        prompt = f"""
        In Maharashtra, India, what is the main growing season for the crop '{crop_name}'?
        Choose only one from: Rabi, Kharif, or Zaid.
        Give only the name of the season in JSON: {{"season": "Rabi"}}
        """
        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "Answer with JSON containing only the season name."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=40,
        )

        raw = response.choices[0].message.content.strip()
        parsed = extract_json_from_text(raw)
        if parsed and "season" in parsed:
            season_text = str(parsed["season"]).strip().capitalize()
            if season_text in ["Rabi", "Kharif", "Zaid"]:
                return season_text

        # Fallback to single-word parse
        season_text = re.search(r"\b(Rabi|Kharif|Zaid)\b", raw, flags=re.IGNORECASE)
        if season_text:
            return season_text.group(0).capitalize()

        return random.choice(["Rabi", "Kharif", "Zaid"]) # Last resort fallback
    except Exception as e:
        print(f"⚠ API error fetching season for {crop_name}: {e}")
        return random.choice(["Rabi", "Kharif", "Zaid"])


# --- Step 6: Check if crop is grown in district (used for filtering) (UNCHANGED) ---
def is_crop_grown_in_district(crop_name, district_name):
    """Uses AI to verify if a crop is typically cultivated in a district."""
    try:
        prompt = f"""
        In Maharashtra, is the crop '{crop_name}' commonly grown in the district '{district_name}'?
        Answer strictly in JSON: {{"grown": "Yes"}} or {{"grown": "No"}} (Only Yes or No).
        """
        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[
                {"role": "system", "content": "Answer only with exactly the JSON object {\"grown\":\"Yes\"} or {\"grown\":\"No\"}."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=30,
        )

        raw = response.choices[0].message.content.strip()
        parsed = extract_json_from_text(raw)
        if parsed and "grown" in parsed:
            return str(parsed["grown"]).strip().lower().startswith("y")
        # fallback parse
        return "yes" in raw.lower()
    except Exception as e:
        print(f"⚠ API error crop-region check ({crop_name}, {district_name}): {e}")
        return True  # assume yes to avoid data gaps


# --- Step 7: Build ALL Factor Maps (API-driven) ---
print("--- Fetching ALL Factors via API (Base WF, Seasons, Irrigation, Districts) ---\n")
crop_water_map = {}
crop_season_map = {}
district_factor_map = {}
irrigation_factor_map = {}

print("1. Fetching Crop Base Water Footprints and Seasons...")
for crop in crops:
    val = get_base_water_footprint_value(crop)
    crop_water_map[crop] = val

    season = get_crop_season(crop)
    crop_season_map[crop] = season

    print(f"  {crop}: Base: {val:.2f} L/kg, Season: {season}")
    time.sleep(1.5)

print("\n2. Fetching District Stress Indices...")
for district in districts:
    factor = get_district_stress_index(district)
    district_factor_map[district] = factor
    print(f"  {district}: Index: {factor:.2f}")
    time.sleep(1.5)

print("\n3. Fetching Irrigation Efficiency Multipliers...")
for i_type in irrigation_types:
    min_f, max_f = get_irrigation_multipliers(i_type)
    irrigation_factor_map[i_type] = (min_f, max_f)
    print(f"  {i_type}: Factors: {min_f:.2f} - {max_f:.2f}")
    time.sleep(1.5)

print("\nAll factor maps prepared via API calls.\n")


# --- Step 8: Generate region-aware dataset (Permutation-based with Factors) ---
records = []
print("Generating dataset from all valid Crop × District × Irrigation_Type combinations...")

for district in districts:
    district_factor = district_factor_map.get(district, 1.0)

    for crop in crops:
        base_value = crop_water_map.get(crop, np.nan)

        # Skip if base value is NaN (API failed to fetch a number)
        if pd.isna(base_value):
             continue

        # Skip if not typically grown in this district (AI check)
        if not is_crop_grown_in_district(crop, district):
            print(f"--- Skipping {crop} in {district} (AI deemed not commonly grown)")
            continue

        for irrigation_type in irrigation_types:

            # 1. Apply Irrigation Factor (Min, Max)
            min_factor, max_factor = irrigation_factor_map.get(irrigation_type, (0.95, 1.05))
            irrigation_multiplier = np.random.uniform(min_factor, max_factor)

            # 2. Apply District Factor

            # 3. Calculate Final Water Requirement
            # Base * Irrigation Factor * District Factor * small random noise
            random_noise = np.random.uniform(0.98, 1.02) # small general noise

            avg_value = (
                base_value
                * irrigation_multiplier
                * district_factor
                * random_noise
            )

            # Assign Category based on factored value
            if avg_value > 3500:
                category = "Very High"
            elif avg_value > 2000:
                category = "High"
            elif avg_value > 1000:
                category = "Medium"
            else:
                category = "Low"

            season = crop_season_map.get(crop, "")

            records.append({
                "State": "Maharashtra",
                "District": district,
                "Crop": crop,
                "Season": season,
                "Irrigation_Type": irrigation_type,
                "District_Stress_Index": round(district_factor, 2),
                "Avg_Water_Requirement(L_per_kg)": round(avg_value, 2),
                "Category": category
            })

    print(f"Completed combinations for {district}")

# --- Step 9: Save the dataset ---
df = pd.DataFrame(records)
print(f"\nTotal valid records generated: {len(df)}")

# Save locally first
local_path = "/content/maharashtra_water_footprint_ai_fully_driven.csv"
df.to_csv(local_path, index=False)
print(f"Saved locally at: {local_path}")

# Copy to Drive
final_path = os.path.join(save_path, "maharashtra_water_footprint_ai_fully_driven.csv")
os.system(f"cp '{local_path}' '{final_path}'")
print(f"Successfully copied to Drive: {final_path}")

print("\nSample Preview:")
# Display the 5 most water-intensive records and the 5 least water-intensive records
print("--- Top 5 Highest Water Footprint Records ---")
print(df.nlargest(5, "Avg_Water_Requirement(L_per_kg)"))
print("\n--- Top 5 Lowest Water Footprint Records ---")
print(df.nsmallest(5, "Avg_Water_Requirement(L_per_kg)"))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Fetching ALL Factors via API (Base WF, Seasons, Irrigation, Districts) ---

1. Fetching Crop Base Water Footprints and Seasons...
⚠ Parse fail (attempt 1) for prompt: Provide the base water footpri...
⚠ Parse fail (attempt 2) for prompt: Provide the base water footpri...
⚠ Parse fail (attempt 3) for prompt: Provide the base water footpri...
⚠ Parse fail (attempt 4) for prompt: Provide the base water footpri...
✖ Failed to obtain valid numerical data after 4 attempts.
  Sugarcane: Base: nan L/kg, Season: Zaid
⚠ Parse fail (attempt 1) for prompt: Provide the base water footpri...
⚠ Parse fail (attempt 2) for prompt: Provide the base water footpri...
⚠ Parse fail (attempt 3) for prompt: Provide the base water footpri...
  Soybean: Base: 1500.00 L/kg, Season: Rabi
⚠ Parse fail (attempt 1) for prompt: Provide the base water footpri...
⚠ Parse fail (attempt 2) 